## Installtion

In [ ]:
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"


## import

In [10]:
from google.colab import drive
import open_dataset_store as ds
import numpy as np
import pandas as pd
import plotly.express as px
import sims as kf_module
from scipy.spatial.transform import Rotation as R

In [ ]:
print('Open Dataset Store Version :',ds.__version__)
drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = ds.quick_start(dataset_path, backend='local')
store.list_entries('experiments')

Open Dataset Store Version : 2.0.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)


In [ ]:
store.list_entries('experiments')

In [8]:
df_1 = store.get_entry_raw_data("experiments", "experiments_0001")
df_2 = store.get_entry_raw_data("experiments", "experiments_0003")

In [12]:
def rotmat_to_quat(R_mat):
    """Convert 3x3 rotation matrix to quaternion [w, x, y, z]"""
    rot = R.from_matrix(R_mat)
    q = rot.as_quat()  # scipy returns [x, y, z, w]
    return np.array([q[3], q[0], q[1], q[2]])  # [w, x, y, z]
    
def DEKF(df,device_prefix):
    accel_cols = [f'{device_prefix}_AX', f'{device_prefix}_AY', f'{device_prefix}_AZ']
    gyro_cols = [f'{device_prefix}_GX', f'{device_prefix}_GY', f'{device_prefix}_GZ']

    accel = df[accel_cols].to_numpy(dtype=float)
    gyro  = df[gyro_cols].to_numpy(dtype=float)
    
    # Normalize accelerometer → unit vector (gravity direction)
    accel_norm = np.linalg.norm(accel, axis=1, keepdims=True)
    accel_norm = np.where(accel_norm < 1e-8, 1.0, accel_norm)
    y_data = accel / accel_norm

    # Filter Configuration
    E = np.array([[0.0, 0.0, -1.0]])
    dt = 0.01
    # Tuning parameters
    Sigma_gyro = (0.02**2) * np.eye(3)   # Gyro noise
    Sigma_bias = (1e-5**2) * np.eye(3)   # Gyro bias random walk
    Sigma_q = np.block([
        [Sigma_gyro, np.zeros((3, 3))],
        [np.zeros((3, 3)), Sigma_bias],
    ])
    Sigma_m = (0.15**2) * np.eye(3)

    kf = kf_module.LinearKF(use_joseph=True, symmetrize=True)
    iekf = kf_module.SO3IMUSensorFusionBiasEKF(kf=kf, inertial_directions=E, dt=dt)
    res = iekf.run_filter(
        R0_plus=np.eye(3),
        b0_plus=np.zeros(3),
        P0=np.diag([1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4]),
        Omegas=gyro,
        Ys=y_data,
        Sigma_q=Sigma_q,
        Sigma_m=Sigma_m
    )
    
    R_est = res["R_plus"]  # (T, 3, 3)
    # Convert to quaternions
    quats = np.array([rotmat_to_quat(R_mat) for R_mat in R_est])
    
    # Output DataFrame
    out = pd.DataFrame()
    if 'Timestamp' in df.columns:
        out['Timestamp'] = df['Timestamp'].values
    else:
        out['Timestamp'] = np.arange(len(df))
    
    out[f'{device_prefix}_qw'] = quats[:, 0]
    out[f'{device_prefix}_qx'] = quats[:, 1]
    out[f'{device_prefix}_qy'] = quats[:, 2]
    out[f'{device_prefix}_qz'] = quats[:, 3]
    
    return out

def Orientation_Estimation(df, device_list=None):
    """
    Run DEKF on multiple devices and combine results.
    """
    if device_list is None:
        # Auto-detect devices from _AX columns
        accel_cols = [col for col in df.columns if col.endswith('_AX')]
        device_list = sorted(set(col.replace('_AX', '') for col in accel_cols))
    
    results = []
    for dev in device_list:
        print(f"Processing {dev} ...")
        try:
            res = DEKF(df, dev)
            results.append(res)
        except Exception as e:
            print(f"    Error on {dev}: {e}")
    
    if not results:
        raise ValueError("No devices processed successfully.")
    
    # Merge all
    final = results[0]
    for r in results[1:]:
        if 'Timestamp' in r.columns:
            final = pd.merge(final, r, on='Timestamp', how='outer')
        else:
            final = pd.concat([final, r], axis=1)
    
    if 'Timestamp' in final.columns:
        final = final.sort_values('Timestamp').reset_index(drop=True)
    
    return final

In [13]:
full_df = Orientation_Estimation(df_1)

Processing device10_IMU1 ...
Processing device10_IMU2 ...
Processing device10_IMU3 ...
Processing device1_IMU1 ...
Processing device1_IMU2 ...
Processing device2_IMU1 ...
Processing device2_IMU2 ...
Processing device3_IMU1 ...
Processing device3_IMU2 ...
Processing device4_IMU1 ...
Processing device4_IMU2 ...
Processing device5_IMU1 ...
Processing device5_IMU2 ...
Processing device5_IMU3 ...
Processing device6_IMU1 ...
Processing device6_IMU2 ...
Processing device6_IMU3 ...
Processing device7_IMU1 ...
Processing device7_IMU2 ...
Processing device7_IMU3 ...
Processing device8_IMU1 ...
Processing device8_IMU2 ...
Processing device8_IMU3 ...
Processing device9_IMU1 ...
Processing device9_IMU2 ...
Processing device9_IMU3 ...


In [14]:
full_df.head()

,Timestamp,device10_IMU1_qw,device10_IMU1_qx,device10_IMU1_qy,device10_IMU1_qz,device10_IMU2_qw,device10_IMU2_qx,device10_IMU2_qy,device10_IMU2_qz,device10_IMU3_qw,...,device9_IMU1_qy,device9_IMU1_qz,device9_IMU2_qw,device9_IMU2_qx,device9_IMU2_qy,device9_IMU2_qz,device9_IMU3_qw,device9_IMU3_qx,device9_IMU3_qy,device9_IMU3_qz
0,1,0.992170,-0.117196,-0.043173,0.000620,0.987778,-0.038861,-0.150756,0.007534,0.988676,...,-0.091353,0.001152,0.997373,0.049109,-0.053196,0.002356,0.991755,-0.014299,-0.127351,-0.000476
1,2,0.980088,-0.186140,-0.069113,0.001607,0.963893,-0.064287,-0.257923,0.015896,0.968001,...,-0.175734,0.002089,0.993782,0.076496,-0.080807,0.003907,0.978514,-0.022549,-0.204938,-0.001200
2,3,0.969210,-0.230928,-0.085413,0.002971,0.939184,-0.084727,-0.331989,0.023185,0.947463,...,-0.252026,0.004377,0.990704,0.094320,-0.097898,0.004939,0.966424,-0.028205,-0.255396,-0.001671
3,4,0.960197,-0.262139,-0.096386,0.003877,0.916568,-0.099888,-0.386029,0.030136,0.929650,...,-0.319629,0.008940,0.988047,0.107412,-0.110445,0.005244,0.956463,-0.033612,-0.289904,-0.001993
4,5,0.952830,-0.284903,-0.104473,0.005581,0.897146,-0.112989,-0.425612,0.034879,0.914308,...,-0.378814,0.013714,0.985779,0.118062,-0.119493,0.004792,0.948036,-0.039060,-0.315743,-0.002971


In [23]:
fig = px.line(
    full_df,
    x='Timestamp',
    width=1500,
    y=['device10_IMU1_qw','device10_IMU1_qx','device10_IMU1_qy','device10_IMU1_qz'],
    title='Device 1 IMU Accelerometer Data Over Time'
)

fig.show(renderer="colab+vscode")